# GEPS要素清单：112-130

In [ ]:
from reki import from_source

In [ ]:
from data_notebook.data import get_data_file

geps_grib2_orig_file_path = get_data_file("cma-geps")

examples = {}
geps_grib2_orig_file_path

## 等压面：相对湿度

### 层次

17 层列表（百帕 (hPa)）：

- 1000
- 975
- 950
- 925
- 900
- 850
- 800
- 750
- 700
- 650
- 600
- 550
- 500
- 400
- 300
- 200
- 100

### 相对湿度

| 要素名 | cemc | wgrib2 | eccodes | GRIB Key | 层次类型 | 层次 | 编号 |
|---|---|---|---|---|---|---|---|
| 相对湿度 | rh | RH | r | (0, 1, 1) | pl | 17层 | 112-128 |

In [ ]:
examples[112] = (
    {
        "parameter": "cedarkit.rh",
        "member": 1,
        "level_type": "pressure",
        "level": 100,
        "level_unit": "hPa",
    },
    {
        "discipline": 0,
        "parameterCategory": 1,
        "parameterNumber": 1,
        "productDefinitionTemplateNumber": 1,
        "typeOfFirstFixedSurface": 100,
        "typeOfSecondFixedSurface": 255,
        "typeOfEnsembleForecast": 3,
        "perturbationNumber": 1,
        "numberOfForecastsInEnsemble": 31,
        "scaledValueOfFirstFixedSurface": 10000,
    },
)

In [ ]:
examples[117] = (
    {
        "parameter": "cedarkit.rh",
        "member": 1,
        "level_type": "pressure",
        "level": 550,
        "level_unit": "hPa",
    },
    {
        "discipline": 0,
        "parameterCategory": 1,
        "parameterNumber": 1,
        "productDefinitionTemplateNumber": 1,
        "typeOfFirstFixedSurface": 100,
        "typeOfSecondFixedSurface": 255,
        "typeOfEnsembleForecast": 3,
        "perturbationNumber": 1,
        "numberOfForecastsInEnsemble": 31,
        "scaledValueOfFirstFixedSurface": 55000,
    },
)

In [ ]:
examples[123] = (
    {
        "parameter": "cedarkit.rh",
        "member": 1,
        "level_type": "pressure",
        "level": 850,
        "level_unit": "hPa",
    },
    {
        "discipline": 0,
        "parameterCategory": 1,
        "parameterNumber": 1,
        "productDefinitionTemplateNumber": 1,
        "typeOfFirstFixedSurface": 100,
        "typeOfSecondFixedSurface": 255,
        "typeOfEnsembleForecast": 3,
        "perturbationNumber": 1,
        "numberOfForecastsInEnsemble": 31,
        "scaledValueOfFirstFixedSurface": 85000,
    },
)

In [ ]:
examples[128] = (
    {
        "parameter": "cedarkit.rh",
        "member": 1,
        "level_type": "pressure",
        "level": 1000,
        "level_unit": "hPa",
    },
    {
        "discipline": 0,
        "parameterCategory": 1,
        "parameterNumber": 1,
        "productDefinitionTemplateNumber": 1,
        "typeOfFirstFixedSurface": 100,
        "typeOfSecondFixedSurface": 255,
        "typeOfEnsembleForecast": 3,
        "perturbationNumber": 1,
        "numberOfForecastsInEnsemble": 31,
        "scaledValueOfFirstFixedSurface": 100000,
    },
)

## 阵风与能见度

### 阵风风速

| 要素名 | cemc | wgrib2 | eccodes | GRIB Key | 层次类型 | 层次 | 编号 |
|---|---|---|---|---|---|---|---|
| 阵风风速 | gust | GUST | i10fg | (0, 2, 22) | heightAboveGround | 10 | 129 |

In [ ]:
examples[129] = (
    {"parameter": "cedarkit.gust", "member": 1},
    {
        "discipline": 0,
        "parameterCategory": 2,
        "parameterNumber": 22,
        "productDefinitionTemplateNumber": 1,
        "typeOfFirstFixedSurface": 103,
        "typeOfSecondFixedSurface": 255,
        "typeOfEnsembleForecast": 3,
        "perturbationNumber": 1,
        "numberOfForecastsInEnsemble": 31,
    },
)

### 能见度

| 要素名 | cemc | wgrib2 | eccodes | GRIB Key | 层次类型 | 层次 | 编号 |
|---|---|---|---|---|---|---|---|
| 能见度 | vis | VIS | vis | (0, 19, 0) | surface | - | 130 |

In [ ]:
examples[130] = (
    {"parameter": "cedarkit.vis", "member": 1},
    {
        "discipline": 0,
        "parameterCategory": 19,
        "parameterNumber": 0,
        "productDefinitionTemplateNumber": 1,
        "typeOfFirstFixedSurface": 1,
        "typeOfSecondFixedSurface": 255,
        "typeOfEnsembleForecast": 3,
        "perturbationNumber": 1,
        "numberOfForecastsInEnsemble": 31,
    },
)

## 核对具名查询与原生键

In [ ]:
with from_source({"kind": "file", "path": str(geps_grib2_orig_file_path)}) as source:
    named = source.sel([query for query, _ in examples.values() if query is not None]).all()
    native = source.sel([{"native": {"grib": keys}} for _, keys in examples.values()]).all()
    fields = {field.metadata["source"]["ordinal"]: field for field in named}
    native_fields = {field.metadata["source"]["ordinal"]: field for field in native}
    assert native_fields.keys() == examples.keys()
    assert fields.keys() == {ordinal for ordinal, (query, _) in examples.items() if query is not None}

    for ordinal, field in fields.items():
        by_native = native_fields[ordinal]
        assert field.identity.parameter == examples[ordinal][0]["parameter"]
        assert field.metadata["source"] == by_native.metadata["source"]
        assert field.read_grib_message() == by_native.read_grib_message()

    with geps_grib2_orig_file_path.open("rb") as original:
        for field in native_fields.values():
            location = field.metadata["source"]
            original.seek(location["offset"])
            assert field.read_grib_message() == original.read(location["length"])